## Solve power flow with MLP load predictions and weather dependent capacity ratings and heat losses

This notebook does the following:

1. Define path to Master file corresponding to TGW year and scenario
2. Solve Power flow
3. Save lines and Xfer data at each timestep (current, voltage, power)

## Initialize

### Import packages

In [2]:
## Import packages
import time
import datetime
import os
import re
import json
import yaml
import joblib
import numpy as np 
import pandas as pd 
import pyarrow as pa
import xarray as xr
import random
import matplotlib.pyplot as plt
import shutil 

from opendssdirect import dss

from src import physics_ops
from src import file_ops
from src import opendss_ops
from src import TGW_ops
from src import input_ops
from src import df_ops

### Define functions

In [ ]:

def build_regional_demand_weather_filename(TGW_years_scenarios):
    """
    Build dynamic output filename based on TGW_years_scenarios.

    Example:
        {
            "1990": ["historical"],
            ...
            "2019": ["historical"],
            "2030": ["rcp45hotter"],
            ...
            "2059": ["rcp45hotter"]
        }

    becomes:
        regional_demand_weather_all_cities_1990_2019_historical_2030_2059_rcp45hotter.joblib
    """

    # Invert year -> scenarios into scenario -> years
    scenario_to_years = {}

    for year, scenarios in TGW_years_scenarios.items():
        year_int = int(year)

        for scenario in scenarios:
            if scenario not in scenario_to_years:
                scenario_to_years[scenario] = []

            scenario_to_years[scenario].append(year_int)

    filename_parts = ["regional_demand_weather_all_cities"]

    # Sort by earliest year for stable filename ordering
    for scenario, years in sorted(
        scenario_to_years.items(),
        key=lambda item: min(item[1])
    ):
        years = sorted(years)

        # Detect contiguous year ranges
        start_year = years[0]
        previous_year = years[0]

        for current_year in years[1:] + [None]:
            if current_year is not None and current_year == previous_year + 1:
                previous_year = current_year
            else:
                # Close current contiguous range
                end_year = previous_year

                if start_year == end_year:
                    filename_parts.append(f"{start_year}_{scenario}")
                else:
                    filename_parts.append(f"{start_year}_{end_year}_{scenario}")

                # Start next range if there is one
                if current_year is not None:
                    start_year = current_year
                    previous_year = current_year

    filename = "_".join(filename_parts) + ".joblib"

    return filename

### Load config file with scenarios and parameters 

In [1]:
config_file_name = 'opendss_config1'; config_path = f"config/{config_file_name}.yaml"; config = input_ops.load_config(config_path)

enable_save = 1  # 1 enable | 0 disable

TGW_years_scenarios = config['TGW_years_scenarios']

demand_mode = config['demand_mode']
    
aggregation_level = config['aggregation_level']

CITY_REGIONS_TO_RUN = config['CITY_REGIONS_TO_RUN']

input_data_dict_name = config['input_data_dict_name']
aggregation_level = config['aggregation_level']
smart_ds_year = config['smart_ds_years'][0]
building_types = config["building_types"]

input_data_training_path = config['input_data_training_path']
CITY_REGIONS_TO_RUN = config['CITY_REGIONS_TO_RUN']
start_month = config['start_month']
end_month = config['end_month']

## Initialize parameters for saving paths
Y_column = config['Y_column']
input_data_prediction_path = config['input_data_prediction_path']
output_path_prediction_str = config['output_data_prediction_path']
output_pf_path = config['output_pf_path']

smart_ds_year = config['smart_ds_years'][0]
smart_ds_load_path = config['smart_ds_load_path'] + f"/{smart_ds_year}" # path to procesed smart-ds resstock data 
   
solution_mode = config['solution_mode']

## Define variables to create list of mdh to run
start_month_mdh = config['start_month_mdh'] 
end_month_mdh = config['end_month_mdh']
top_percent_mdh = config['top_percent_mdh']

## Load dictionary, sort by total city aggregated buildings demand, extract mdh of top % load hours
regional_demand_weather_filename = build_regional_demand_weather_filename(TGW_years_scenarios)
city = "all_cities"
regional_demand_weather_path = (
    smart_ds_load_path
    + f"/{city}/aggregated_demand/"
    + regional_demand_weather_filename
)
regional_demand_weather_ampacity_all_cities = joblib.load(regional_demand_weather_path)
regional_demand_weather_ampacity_all_cities_sorted = df_ops.sort_nested_dict_dfs(regional_demand_weather_ampacity_all_cities, "aggregated_predicted_buildings_total_kw", ascending=False)

top_n_hours = int(np.ceil(8760*top_percent_mdh/100)) # calculate top city demand hours to run (top_percent_mdh% of hours of the year)
## Define start and end load hours to run
start_row_percent = config['start_row_percent']
start_row_idx = int(np.ceil(8760*start_row_percent/100)) # index from which to start loop of load hours
# start_row_idx = 0 # index from which to start loop of load hours
end_row_idx = top_n_hours 

# Solar / battery SMART-DS scenario parameters
solar_share = config.get("solar_share", "none")
battery_share = config.get("battery_share", "none")

solar_battery_scenario_folder = input_ops.build_solar_battery_scenario_folder(
    solar_share=solar_share,
    battery_share=battery_share,
)

print(f"solar_share: {solar_share}\n battery_share: {battery_share}\n solar_battery_scenario_folder: {solar_battery_scenario_folder}")

print(f"enable_save:{enable_save} \nsolution_mode: {solution_mode} \nTGW_years_scenarios:{TGW_years_scenarios} \nCITY_REGIONS_TO_RUN: {CITY_REGIONS_TO_RUN}  \n demand mode: {demand_mode}")

print(f"\n\ntop_percent_mdh: {top_percent_mdh}%, top_n_hours: {top_n_hours}, start_row_idx:{start_row_idx} ({start_row_percent}%), end_row_idx:{end_row_idx} ({top_percent_mdh}%) \n")

## Run power flow 
Note: should take about 40min for all scenarios and 3 TGW year-scenario combinations

In [1]:
start_time = time.time()

for TGW_weather_year, TGW_scenarios in TGW_years_scenarios.items():
    for TGW_scenario in TGW_scenarios:
        print(f"--- Starting scenario: {TGW_weather_year} {TGW_scenario} ---\n")
        year = TGW_weather_year; 
        # --- Iterate over all selected regions --- 
        for city, regions in CITY_REGIONS_TO_RUN.items():
            
            ## create list of mdh for top % hours
            df_city = regional_demand_weather_ampacity_all_cities_sorted[(TGW_weather_year, TGW_scenario)][city]   # load regional demand data to later create list of mdh for top % hours  
            list_of_mdh = df_ops.get_top_n_mdh(df_city, top_n_hours, start_month_mdh, end_month_mdh)

            for region in regions:
                print(f"--- Starting scenario: {city} {region} ---\n")
                # --- Iterate over all selected month-day-hour (mdh) --- 
                
                # --- Initialize region-level containers ---
                line_dfs = []
                transformer_dfs = []
                pvsystems_dfs = []

                pf_convergence_records = []
                all_pf_converged = True
                n_converged_mdh = 0
                n_failed_mdh = 0
                
                # Create path to regional master file 
                region_path = (
                    f"main_folder/SMART-DS/v1.0/"
                    f"{smart_ds_year}/{city}/{region}/scenarios/"
                    f"{solar_battery_scenario_folder}/opendss_no_loadshapes"
                )

                
                dict_bus_coord = opendss_ops.create_dict_bus_coord(region_path)
                
                for row_i in range(start_row_idx, end_row_idx):
                    mdh = list_of_mdh[row_i]; m,d,h = mdh
                    print(f"--- Starting Index: {row_i}, mdh: {mdh} ---\n")

                    # --- Redirect opendss to new master file  ---
                    dss.Basic.ClearAll() 

                    # --- Redirect opendss to TGW scenario master file  ---
                    new_master_dir = os.path.join(region_path, "predicted_master_files", TGW_scenario, TGW_weather_year)
                    dss.Command(f'Redirect "{new_master_dir}/Master_{TGW_scenario}_{TGW_weather_year}_{m}_{d}_{h}.dss"') # Direct opendss engine to master file 
                    
                    dss.Command("Set MaxControlIter=100")
                    dss.Command("Set MaxIterations=100")
                    
                    # --- Solve power flow ---
                    pf_start_time = time.time()
                    dss.Solution.Solve()

                    pf_iterations = dss.Solution.Iterations()
                    pf_converged = bool(dss.Solution.Converged())

                    pf_end_time = time.time()
                    pf_runtime_minutes = (pf_end_time - pf_start_time) / 60

                    pf_convergence_records.append({
                        "TGW_weather_year": TGW_weather_year,
                        "TGW_scenario": TGW_scenario,
                        "smart_ds_year": smart_ds_year,
                        "city": city,
                        "region": region,
                        "solar_share": solar_share,
                        "battery_share": battery_share,
                        "solar_battery_scenario_folder": solar_battery_scenario_folder,
                        "row_i": row_i,
                        "month": m,
                        "day": d,
                        "hour": h,
                        "converged": pf_converged,
                        "iterations": pf_iterations,
                        "runtime_minutes": pf_runtime_minutes,
                    })

                    if pf_converged:
                        print(
                            f"--- Power flow solution converged successfully "
                            f"after {pf_iterations} iterations ---\n"
                        )
                        n_converged_mdh += 1
                    else:
                        print(
                            "\n"
                            "!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!\n"
                            "WARNING: POWER FLOW DID NOT CONVERGE\n"
                            f"Scenario: {TGW_weather_year} {TGW_scenario}, {city} {region}\n"
                            f"Solar/battery folder: {solar_battery_scenario_folder}\n"
                            f"row_i: {row_i}, mdh: {mdh}\n"
                            "Skipping extraction for this MDH.\n"
                            "!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!\n"
                        )
                        all_pf_converged = False
                        n_failed_mdh += 1
                        print("PF solve runtime:", pf_runtime_minutes, "minutes")
                        continue

                    print("PF solve runtime:", pf_runtime_minutes, "minutes")
                   
                    # --- Extract line, transformer, and PVSystem data (physical properties, loading, voltage etc.) ---
                    line_df = opendss_ops.extract_line_information(dict_bus_coord=dict_bus_coord,weather_year=year,m=m,d=d,h=h,row_i=row_i, enable_low_memory=True)
                    transformer_df = opendss_ops.extract_transformer_information(dict_bus_coord=dict_bus_coord,weather_year=year,m=m,d=d,h=h,row_i=row_i, enable_low_memory=True)
                    pvsystems_df = opendss_ops.extract_pvsystems_information(dict_bus_coord=dict_bus_coord,weather_year=year,m=m,d=d,h=h,row_i=row_i,decimals=6,extraction_mode="minimal", enable_low_memory=True)
 
                    line_dfs.append(line_df)
                    transformer_dfs.append(transformer_df)
                    pvsystems_dfs.append(pvsystems_df)
                
                # --- Concatenate extracted data for this region ---
                if len(line_dfs) > 0:
                    all_line_df = pd.concat(line_dfs, ignore_index=True)
                else:
                    all_line_df = pd.DataFrame()

                if len(transformer_dfs) > 0:
                    all_transformer_df = pd.concat(transformer_dfs, ignore_index=True)
                else:
                    all_transformer_df = pd.DataFrame()

                if len(pvsystems_dfs) > 0:
                    non_empty_pv_dfs = [df for df in pvsystems_dfs if df is not None and not df.empty]
                    if len(non_empty_pv_dfs) > 0:
                        all_pvsystems_df = pd.concat(non_empty_pv_dfs, ignore_index=True)
                    else:
                        all_pvsystems_df = pd.DataFrame()
                else:
                    all_pvsystems_df = pd.DataFrame()

                print(
                    f"\n--- Convergence summary for {TGW_weather_year} {TGW_scenario}, "
                    f"{city} {region}, {solar_battery_scenario_folder} ---\n"
                    f"Converged MDHs: {n_converged_mdh}\n"
                    f"Failed MDHs: {n_failed_mdh}\n"
                    f"All PF converged: {all_pf_converged}\n"
                )

                if n_converged_mdh == 0:
                    print(
                        "\n"
                        "!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!\n"
                        "WARNING: NO MDHs CONVERGED FOR THIS REGION/SCENARIO.\n"
                        "The saved line, transformer, and PVSystem DataFrames will be empty.\n"
                        "Check the metadata convergence records for details.\n"
                        "!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!\n"
                    )

                
                # --- Save results per region ---
                lines_file_name = f"lines_top_{start_row_percent}_{top_percent_mdh}_percent"
                transformers_file_name = f"transformers_top_{start_row_percent}_{top_percent_mdh}_percent"
                pvsystems_file_name = f"pvsystems_top_{start_row_percent}_{top_percent_mdh}_percent"
                metadata_file_name = f"metadata_top_{start_row_percent}_{top_percent_mdh}_percent"

                # --- Initialize dictionaries ---
                lines_dict = {}
                transformers_dict = {}
                pvsystems_dict = {}

                # Initialize inner dictionary per TGW scenario
                lines_dict[(TGW_weather_year, TGW_scenario)] = {}
                transformers_dict[(TGW_weather_year, TGW_scenario)] = {}
                pvsystems_dict[(TGW_weather_year, TGW_scenario)] = {}

                # Add DataFrames to second-level SMART-DS key
                smartds_region_key = (smart_ds_year, city, region)

                lines_dict[(TGW_weather_year, TGW_scenario)][smartds_region_key] = all_line_df
                transformers_dict[(TGW_weather_year, TGW_scenario)][smartds_region_key] = all_transformer_df
                pvsystems_dict[(TGW_weather_year, TGW_scenario)][smartds_region_key] = all_pvsystems_df

                if enable_save == 1:
                    # Save dictionaries as joblib files
                    predictions_dir = os.path.join(output_pf_path,city,region,TGW_scenario,TGW_weather_year,solar_battery_scenario_folder)
                    os.makedirs(predictions_dir, exist_ok=True)
                      
                    # --- Save dictionaries as joblib files ---
                    file_suffix = "" if all_pf_converged else "_did_not_converge"

                    joblib.dump(
                        lines_dict,
                        os.path.join(predictions_dir, f"{lines_file_name}{file_suffix}.joblib"),
                    )

                    joblib.dump(
                        transformers_dict,
                        os.path.join(predictions_dir, f"{transformers_file_name}{file_suffix}.joblib"),
                    )

                    joblib.dump(
                        pvsystems_dict,
                        os.path.join(predictions_dir, f"{pvsystems_file_name}{file_suffix}.joblib"),
                    )    

                    # --- Define metadata ---
                    timestamp = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

                    metadata = {
                        "changes": "demand_and_solar_battery_scenario",
                        "solution_mode": solution_mode,
                        "master_file_option": "TGW",
                        "smart_ds_year": smart_ds_year,
                        "TGW_weather_year": TGW_weather_year,
                        "TGW_scenario": TGW_scenario,
                        "city": city,
                        "region": region,
                        "solar_share": solar_share,
                        "battery_share": battery_share,
                        "solar_battery_scenario_folder": solar_battery_scenario_folder,
                        "top_percent_mdh": top_percent_mdh,
                        "start_row_percent": start_row_percent,
                        "start_row_idx": start_row_idx,
                        "end_row_idx": end_row_idx,
                        "n_requested_mdh": end_row_idx - start_row_idx,
                        "n_converged_mdh": n_converged_mdh,
                        "n_failed_mdh": n_failed_mdh,
                        "all_pf_converged": all_pf_converged,
                        "pf_convergence_records": pf_convergence_records,
                        "saved_file_suffix": file_suffix,
                        "timestamp": timestamp,
                    }

                    metadata_file = os.path.join(predictions_dir, f"{metadata_file_name}.json")

                    with open(metadata_file, "w") as f:
                        json.dump(metadata, f, indent=4)

                    if all_pf_converged:
                        print(
                            f"---- Results for scenario {city} {region} saved successfully "
                            f"in {predictions_dir} ----\n"
                        )
                    else:
                        print(
                            "\n"
                            "!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!\n"
                            f"WARNING: Results for {city} {region} were saved with suffix "
                            f"{file_suffix!r} because at least one MDH did not converge.\n"
                            f"See metadata file for convergence details:\n{metadata_file}\n"
                            "!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!\n"
                        )
    
end_time = time.time(); print("Total runtime:", (end_time - start_time) / 60, "minutes")

### Print bus voltages 

In [2]:
buses = dss.Circuit.AllBusNames()
for bus in buses:
    dss.Circuit.SetActiveBus(bus)
    voltages = dss.Bus.puVmagAngle()
    voltages_pu = voltages[::2]
    for v in voltages_pu:
        if v < 0.9 or v > 1.1:
            print(f"Voltage warning at {bus}: {v:.3f} pu")

### Print circuit loads

In [3]:
# Get total losses
losses = dss.Circuit.Losses()
total_loss_kw = losses[0] / 1000  # Convert watts to kW          
# Get total circuit power 
total_circuit_power = dss.Circuit.TotalPower() #  sum of all the powers in the first terminal of each source (Vsource and Isource)
total_circuit_power_kw = - total_circuit_power[0]  # Real power in kW          
# Total load
total_circuit_Loads = total_circuit_power_kw -total_loss_kw

print(f'Total circuit load: {total_circuit_Loads} kw')

### Print line and transformer data

In [2]:
top_10_loading = all_line_df.nlargest(10, 'Loading [%]')
display(top_10_loading)
top_10_loading = all_transformer_df.nlargest(10, 'Loading [%]')
display(top_10_loading)